In [13]:
import pandas as pd
import pandas as pd
import jax
import jax.numpy as jnp
from jax import jit, grad, vmap, pmap
import math
import orbax.checkpoint as ocp
from functools import partial
import os
import gc


jax.config.update("jax_disable_jit", False)
jax.config.update("jax_platform_name", "cpu")

os.environ["JAX_LOG_LEVEL"] = "0" 

jax.config.update("jax_default_matmul_precision", "default") 


gc.collect()

print("aye 1")

gold_second_futures = pd.read_csv('GC2026_second.csv')

gold_second_futures['time'] = pd.to_datetime(
    gold_second_futures['time']
)

gold_second_futures['timestamp'] = (
    gold_second_futures['time'].astype('int64') // 10**9
)


gold_second_futures['timestamp'] = (
    (gold_second_futures['timestamp'] - gold_second_futures['timestamp'].min())/ (gold_second_futures['timestamp'].max() - gold_second_futures['timestamp'].min())
)


X = gold_second_futures[
    ["timestamp", "volume"]
].to_numpy()

y = gold_second_futures[
    ["close"]
].to_numpy()   # (N, 1)

train_data = int(len(X) * 0.8)

X_train = X[:train_data]
y_train = y[:train_data]

X_test = X[train_data:]
y_test = y[train_data:]


sequence_length = 10
batch_size = 32

print("aye 2")

def create_sequences(X, y, sequence_length):
    X_sequences = []
    y_sequences = []

    for i in range(len(X) - sequence_length + 1):
        X_sequences.append(
            X[i:i + sequence_length]
        )

        y_sequences.append(
            y[i:i + sequence_length]
        )

    return (
        jnp.asarray(X_sequences, dtype=jnp.bfloat16),
        jnp.asarray(y_sequences, dtype=jnp.bfloat16)
    )


X_train_seq, y_train_seq = create_sequences(
    X_train,
    y_train,
    sequence_length
)

X_test_seq, y_test_seq = create_sequences(
    X_test,
    y_test,
    sequence_length
)

print("aye 3")




aye 1
aye 2
aye 3


In [14]:
print("X_train_seq",X_train_seq[0].dtype)
print("y_train_seq",y_train_seq[0].dtype)
print("X_test_seq",X_test_seq[0].dtype)
print("y_test_seq",y_test_seq[0].dtype)


X_train_seq bfloat16
y_train_seq bfloat16
X_test_seq bfloat16
y_test_seq bfloat16


In [15]:



# def adam_optimizer(curr_weights, curr_gradients,bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, iter, layer, is_bias=False):
#     learning_rate = 1e-3
#     beta_1 = 0.9
#     beta_2 = 0.999
#     epsilon = 1e-8

#     if is_bias:
#         bias_moment_history_for_layer = bias_moment_1_history[layer] if bias_moment_1_history[layer] is not None else jnp.zeros(curr_weights.shape)
#         moment_1_curr = (beta_1*bias_moment_history_for_layer) + (1-beta_1) * curr_gradients
#         bias_moment_1_history.at[layer].set(moment_1_curr)
#     else:
#         moment_1_history_for_layer = moment_1_history[layer] if moment_1_history[layer] is not None else jnp.zeros(curr_weights.shape)
#         moment_1_curr = (beta_1*moment_1_history_for_layer) + (1-beta_1) * curr_gradients
#         moment_1_history.at[layer].set(moment_1_curr)

#     hat_moment_1 = moment_1_curr/(1-beta_1**iter)

#     if is_bias:
#         bias_moment_2_history_for_layer = bias_moment_2_history[layer] if bias_moment_2_history[layer] is not None else jnp.zeros(curr_weights.shape)
#         moment_2_curr = beta_2*bias_moment_2_history_for_layer + (1-beta_2) * curr_gradients**2
#         bias_moment_2_history.at[layer].set(moment_2_curr)
#     else:
#         moment_2_history_for_layer = moment_2_history[layer] if moment_2_history[layer] is not None else jnp.zeros(curr_weights.shape)
#         moment_2_curr = beta_2*moment_2_history_for_layer + (1-beta_2) * curr_gradients**2
#         moment_2_history.at[layer].set(moment_2_curr)

#     hat_moment_2 = moment_2_curr/(1-beta_2**iter)

#     return ((learning_rate/(jnp.sqrt(hat_moment_2)+epsilon)) * hat_moment_1), bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history

In [16]:
def adam_optimizer(curr_weights, curr_gradients, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, iter, layer, is_bias=False):
    learning_rate = 1e-3
    beta_1 = 0.9
    beta_2 = 0.999
    epsilon = 1e-8

    # Prevent division by zero if cum_iters starts at 0
    safe_iter = jnp.maximum(1, iter)

    if is_bias:
        moment_1_history_for_layer = bias_moment_1_history[layer]
        moment_1_curr = (beta_1 * moment_1_history_for_layer) + (1.0 - beta_1) * curr_gradients
        # Create a new list copy instead of mutating in-place for JAX pure functional compatibility
        bias_moment_1_history = [m if i != layer else moment_1_curr for i, m in enumerate(bias_moment_1_history)]
    else:
        moment_1_history_for_layer = moment_1_history[layer]
        moment_1_curr = (beta_1 * moment_1_history_for_layer) + (1.0 - beta_1) * curr_gradients
        moment_1_history = [m if i != layer else moment_1_curr for i, m in enumerate(moment_1_history)]

    hat_moment_1 = moment_1_curr / (1.0 - beta_1 ** safe_iter)

    if is_bias:
        moment_2_history_for_layer = bias_moment_2_history[layer]
        moment_2_curr = beta_2 * moment_2_history_for_layer + (1.0 - beta_2) * (curr_gradients ** 2)
        bias_moment_2_history = [m if i != layer else moment_2_curr for i, m in enumerate(bias_moment_2_history)]
    else:
        moment_2_history_for_layer = moment_2_history[layer]
        moment_2_curr = beta_2 * moment_2_history_for_layer + (1.0 - beta_2) * (curr_gradients ** 2)
        moment_2_history = [m if i != layer else moment_2_curr for i, m in enumerate(moment_2_history)]

    hat_moment_2 = moment_2_curr / (1.0 - beta_2 ** safe_iter)

    adam_optimized_change = (learning_rate / (jnp.sqrt(hat_moment_2) + epsilon)) * hat_moment_1

    return adam_optimized_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history

In [17]:
def single_tanh(act):
  res = (jnp.e**act - jnp.e**-act)/(jnp.e**act + jnp.e**-act)
  return res

single_tanh_jit = jit(single_tanh)

def tanh_activation(pre_activations):
  pre_act_shape = pre_activations.shape
  reshaped_pre_acts = jnp.reshape(pre_activations,(-1,))
  tanh_acts = vmap(single_tanh_jit)(reshaped_pre_acts)
  return jnp.asarray(jnp.reshape(tanh_acts,pre_act_shape),dtype=jnp.bfloat16)


In [18]:
def meanAbsoluteLoss(predictions,true_labels):
  return jnp.mean(jnp.abs(predictions-true_labels))

def meanAbsoluteLossDerivation(predictions,true_labels):
  sequence_length = 10
  batch_size = 32
  return jnp.sign(predictions-true_labels)/(sequence_length * batch_size)


In [19]:
@jax.jit
def forward(layer_weights,inputs, biases):
  # layer_idx 0 for rnn layer 1, 1 for rnn layer 2, 2 for dense layer 1 which is also output layer
  sequence_length = inputs.shape[1]
  concatted_inputs = jnp.zeros((sequence_length,inputs.shape[0],66),dtype=jnp.bfloat16)
  layer_0_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[0].shape[1]),dtype=jnp.bfloat16)
  layer_1_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[1].shape[1]),dtype=jnp.bfloat16)
  layer_2_activations = jnp.zeros((sequence_length,inputs.shape[0],layer_weights[2].shape[1]),dtype=jnp.bfloat16)
  l0prev=jnp.zeros((inputs.shape[0],64),dtype=jnp.bfloat16)
  l1prev=jnp.zeros((inputs.shape[0],32),dtype=jnp.bfloat16)

  sequence_length_arange = jnp.arange(sequence_length)

  for i in sequence_length_arange:

    inputs_for_layer_0 = jnp.concatenate([inputs[:,i,:],l0prev],dtype=jnp.bfloat16,axis=1)
      
    temp_layer_0_activation = tanh_activation((inputs_for_layer_0@layer_weights[0]) +biases[0])

    inputs_for_layer_1 = jnp.concatenate([temp_layer_0_activation,l1prev],dtype=jnp.bfloat16,axis=1)

    temp_layer_1_activation = tanh_activation((inputs_for_layer_1@layer_weights[1]) +biases[1])
    temp_layer_2_activation = (temp_layer_1_activation@layer_weights[2]) +biases[2]
      
    layer_0_activations.at[i].set(jnp.asarray(temp_layer_0_activation,dtype=jnp.bfloat16))
    layer_1_activations.at[i].set(jnp.asarray(temp_layer_1_activation,dtype=jnp.bfloat16))
    layer_2_activations.at[i].set(jnp.asarray(temp_layer_2_activation,dtype=jnp.bfloat16))
    concatted_inputs.at[i].set(jnp.asarray(inputs_for_layer_0,dtype=jnp.bfloat16))

    l0prev,l1prev=temp_layer_0_activation,temp_layer_1_activation
  return concatted_inputs,layer_0_activations, layer_1_activations, layer_2_activations





In [20]:
@partial(jax.jit, static_argnames=['sequence_length'])
def backward(
    layer_weights,
    sequence_length,
    concatted_inputs,
    true_labels,
    layer_0_activations,
    layer_1_activations,
    layer_2_activations
    ):

  accumulated_layer_0_weights_changes = None
  accumulated_layer_1_weights_changes = None
  accumulated_layer_2_weights_changes = None

  accumulated_layer_0_bias_changes = None
  accumulated_layer_1_bias_changes = None
  accumulated_layer_2_bias_changes = None

  layer_0_next_timestep_delta = None
  layer_1_next_timestep_delta = None

  for i in range(sequence_length - 1, -1, -1):

    try:
        preds = jnp.asarray(layer_2_activations[i],dtype=jnp.bfloat16)
        tru_labels = jnp.asarray(true_labels[:,i],dtype=jnp.bfloat16)
        mae_gradient =  meanAbsoluteLossDerivation(preds, tru_labels)
        layer_2_weights_change = layer_1_activations[i].T @ mae_gradient   
        
        layer_2_bias_change = mae_gradient

    
        layer_1_activations_gradients = (
                        mae_gradient
                        @
                        layer_weights[2].T
                        
                    )


        layer_1_pre_activations_gradients = (
                layer_1_activations_gradients
                * (1-layer_1_activations[i]**2)
                
        ) # this is also the gradient for bias

        layer_1_bias_change = jnp.copy(layer_1_pre_activations_gradients)

        concatenated_layer_1_inputs = jnp.concat([layer_0_activations[i], (layer_1_activations[i-1] if i > 0 else jnp.zeros_like(layer_1_activations[i]))], axis=1)

        layer_1_weights_change = (
           concatenated_layer_1_inputs.T
           @
           layer_1_pre_activations_gradients
           )

        layer_0_activations_gradients = (
                    layer_1_pre_activations_gradients
                    @
                    layer_weights[1][:64,:].T
                )
    
        layer_0_pre_activations_gradients = (
            layer_0_activations_gradients
            * (1-layer_0_activations[i]**2)
            
        ) 

        layer_0_bias_change = jnp.copy(layer_0_pre_activations_gradients)


        layer_0_weights_change = (
                concatted_inputs[i].T
                @
                layer_0_pre_activations_gradients 
            )
        
            
        if i < sequence_length-1:
            next_time_steps_layer_0_latent_repr_weights = layer_weights[0][2:,:] 
            layer_0_last_timestep_activations_gradients = (
               layer_0_next_timestep_delta @ next_time_steps_layer_0_latent_repr_weights.T
            )
            layer_0_last_timestep_pre_activations_gradients = (
               layer_0_last_timestep_activations_gradients * (1-layer_0_activations[i]**2)
            )

            layer_0_bias_change = layer_0_bias_change.at[:,:].add(layer_0_last_timestep_pre_activations_gradients)

            layer_0_last_timestep_weights_change = (
               concatted_inputs[i].T
               @
               layer_0_last_timestep_pre_activations_gradients
            )

            next_time_steps_layer_1_latent_repr_weights = layer_weights[1][64:,:]
            layer_1_last_timestep_activations_gradients = (
                layer_1_next_timestep_delta @ next_time_steps_layer_1_latent_repr_weights.T
            )
            layer_1_last_timestep_pre_activations_gradients = (
                layer_1_last_timestep_activations_gradients * (1-layer_1_activations[i]**2)
            )

            layer_1_bias_change = layer_1_bias_change.at[:,:].add(layer_1_last_timestep_pre_activations_gradients)

            concatenated_layer_1_input_for_last_timestep = jnp.concat([layer_0_activations[i], (layer_1_activations[i-1] if i > 0 else jnp.zeros_like(layer_1_activations[i]))], axis=1)

            layer_1_last_timestep_weights_change = (
                concatenated_layer_1_input_for_last_timestep.T
                @
                layer_1_last_timestep_pre_activations_gradients
            )

            layer_1_weights_change = layer_1_last_timestep_weights_change.at[:,:].add(layer_1_weights_change)
            layer_0_weights_change = layer_0_last_timestep_weights_change.at[:,:].add(layer_0_weights_change)

            total_delta_layer_0 = layer_0_pre_activations_gradients + layer_0_last_timestep_pre_activations_gradients
            total_delta_layer_1 = layer_1_pre_activations_gradients + layer_1_last_timestep_pre_activations_gradients
        else:
           total_delta_layer_0 = layer_0_pre_activations_gradients
           total_delta_layer_1 = layer_1_pre_activations_gradients


        layer_0_next_timestep_delta = total_delta_layer_0
        layer_1_next_timestep_delta = total_delta_layer_1

        accumulated_layer_0_weights_changes = (
            layer_0_weights_change
            if accumulated_layer_0_weights_changes is None
            else accumulated_layer_0_weights_changes.at[:,:].add(layer_0_weights_change)
        )
        accumulated_layer_1_weights_changes = (
            layer_1_weights_change
            if accumulated_layer_1_weights_changes is None
            else accumulated_layer_1_weights_changes.at[:,:].add(layer_1_weights_change)
        )
        accumulated_layer_2_weights_changes = (
            layer_2_weights_change
            if accumulated_layer_2_weights_changes is None
            else accumulated_layer_2_weights_changes.at[:,:].add(layer_2_weights_change)
        )

        accumulated_layer_0_bias_changes = (
            layer_0_bias_change
            if accumulated_layer_0_bias_changes is None
            else accumulated_layer_0_bias_changes.at[:,:].add(layer_0_bias_change)
        )
        accumulated_layer_1_bias_changes = (
            layer_1_bias_change
            if accumulated_layer_1_bias_changes is None
            else accumulated_layer_1_bias_changes.at[:,:].add(layer_1_bias_change)
        )
        accumulated_layer_2_bias_changes = (
            layer_2_bias_change
            if accumulated_layer_2_bias_changes is None
            else accumulated_layer_2_bias_changes.at[:,:].add(layer_2_bias_change)
        )


    except Exception as e:
        print(str(e))
        raise e

  return (
      accumulated_layer_0_weights_changes,
      accumulated_layer_1_weights_changes,
      accumulated_layer_2_weights_changes,
      accumulated_layer_0_bias_changes,
      accumulated_layer_1_bias_changes,
      accumulated_layer_2_bias_changes
  )




In [21]:
def he_initialization(layer_shapes):
    key = jax.random.key(1337)
    key, w_key = jax.random.split(key)
    layer_weights=[]
    layer_biases=[]

    # Define a He/Kaiming normal initializer (excellent for ReLU activations)
    initializer = jax.nn.initializers.he_normal()

    # Initialize the array
    for i, shape in enumerate(layer_shapes):
        layer_weights.append(initializer(w_key, shape, jnp.bfloat16))
        layer_biases.append(jnp.zeros(shape[-1],dtype=jnp.bfloat16))

    return layer_weights,layer_biases

In [22]:

@jax.jit(static_argnums=(0,1,2))
def training_loop(batch_size, iters_per_epoch, epochs):
  layer_weights_shapes = [(66,64),(96,32),(32,1)]
  layer_weights,layer_biases = he_initialization(layer_weights_shapes)

  moment_1_history = [jnp.zeros_like(w) for w in layer_weights]
  moment_2_history = [jnp.zeros_like(w) for w in layer_weights]
  bias_moment_1_history = [jnp.zeros_like(b) for b in layer_biases]
  bias_moment_2_history = [jnp.zeros_like(b) for b in layer_biases]


  def train_step(carry_state, iter):
  # try:
    cum_iters, epoch, layer_weights, layer_biases, loss, moment_1_history, moment_2_history, bias_moment_1_history, bias_moment_2_history = carry_state
    batch_start = iter * batch_size
    batch_end = (iter + 1) * batch_size
    # slice_size = batch_end - batch_start
    print("lasagnae",epoch, iter, batch_start, batch_end)

    batch_X = jax.lax.dynamic_slice_in_dim(X_train_seq, batch_start, batch_size, axis=0)
    batch_y = jax.lax.dynamic_slice_in_dim(y_train_seq, batch_start, batch_size, axis=0)
    concatted_inputs,layer_0_activations, layer_1_activations, layer_2_activations = forward(layer_weights,batch_X,layer_biases)

    # Fix: Transpose predictions from (Seq, Batch, 1) to (Batch, Seq, 1) to match batch_y
    preds_transposed = jnp.transpose(layer_2_activations, (1, 0, 2))
    loss = meanAbsoluteLoss(preds_transposed, batch_y)

    print("loss this iteration", loss)

    (layer_0_weights_change, 
    layer_1_weights_change, 
    layer_2_weights_change, 
    layer_0_bias_changes, 
    layer_1_bias_changes, 
    layer_2_bias_changes ) = backward(
      layer_weights,
      sequence_length,
      concatted_inputs,
      batch_y,
      layer_0_activations, 
      layer_1_activations, 
      layer_2_activations
      )
    learning_rate = 0.001

    #normalizing the gradients to prevent exploding gradients
    global_sq_sum = (
      jnp.sum(jnp.square(layer_0_weights_change)) +
      jnp.sum(jnp.square(layer_1_weights_change)) +
      jnp.sum(jnp.square(layer_2_weights_change))
    )

    global_norm = jnp.sqrt(global_sq_sum)

    max_norm = 1.0  # Set the maximum allowed L2 norm for the gradients
    clip_coefficient = jnp.minimum(max_norm, max_norm / (global_norm + 1e-6))

    layer_0_weights_change = layer_0_weights_change * clip_coefficient
    layer_1_weights_change = layer_1_weights_change * clip_coefficient 
    layer_2_weights_change = layer_2_weights_change * clip_coefficient

    layer_0_adam_optimized_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history = adam_optimizer(layer_weights[0], layer_0_weights_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, cum_iters, 0)
    layer_1_adam_optimized_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history = adam_optimizer(layer_weights[1], layer_1_weights_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, cum_iters, 1)
    layer_2_adam_optimized_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history = adam_optimizer(layer_weights[2], layer_2_weights_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, cum_iters, 2)
    layer_0_bias_adam_optimized_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history = adam_optimizer(layer_biases[0], layer_0_bias_changes, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, cum_iters, 0, is_bias=True)
    layer_1_bias_adam_optimized_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history = adam_optimizer(layer_biases[1], layer_1_bias_changes, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, cum_iters, 1, is_bias=True)
    layer_2_bias_adam_optimized_change, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history = adam_optimizer(layer_biases[2], layer_2_bias_changes, bias_moment_1_history, moment_1_history, bias_moment_2_history, moment_2_history, cum_iters, 2, is_bias=True)
    
    
    layer_weights[0] = layer_weights[0] - layer_0_adam_optimized_change
    layer_weights[1] = layer_weights[1] - layer_1_adam_optimized_change
    layer_weights[2] = layer_weights[2] - layer_2_adam_optimized_change
    layer_biases[0] = layer_biases[0] - layer_0_bias_adam_optimized_change
    layer_biases[1] = layer_biases[1] - layer_1_bias_adam_optimized_change
    layer_biases[2] = layer_biases[2] - layer_2_bias_adam_optimized_change
    cum_iters+=1

    updated_weights = [w.astype(jnp.bfloat16) for w in layer_weights]
    updated_biases = [b.astype(jnp.bfloat16) for b in layer_biases]
    updated_loss = loss.astype(jnp.bfloat16)

    new_carry_state = (cum_iters, epoch, updated_weights, updated_biases, updated_loss, moment_1_history, moment_2_history, bias_moment_1_history, bias_moment_2_history) 
    return new_carry_state, updated_loss

    # except Exception as e:
    #   print("Error in train_step:", e)
    #   raise e
    
  cum_iters = 0
  epoch_idxs = jnp.arange(epochs)

  for epoch in epoch_idxs:
    range_arr = jnp.arange(iters_per_epoch)
    epoch = epoch.astype(jnp.int32)
    layer_weights = [w.astype(jnp.bfloat16) for w in layer_weights]
    layer_biases = [b.astype(jnp.bfloat16) for b in layer_biases]
    init_loss = jnp.zeros((), dtype=jnp.bfloat16)
    carry_state = (cum_iters, epoch, layer_weights, layer_biases, init_loss, moment_1_history, moment_2_history, bias_moment_1_history, bias_moment_2_history)
    final_carry_state, updated_loss = jax.lax.scan(train_step, carry_state, range_arr)
    cum_iters = final_carry_state[0]
    layer_weights = final_carry_state[2]
    layer_biases = final_carry_state[3]
    epoch_loss = final_carry_state[4]
    print("epoch_loss", updated_loss)
    

  return final_carry_state, updated_loss




In [23]:
batch_size = 32
len_train_samples = len(X_train_seq)
iters_per_epoch = math.ceil(len_train_samples / batch_size)
epochs=10

print(epochs*iters_per_epoch, "total iterations")

final_carry_state, updated_loss = training_loop(batch_size, iters_per_epoch, epochs)

jax.block_until_ready(final_carry_state)  # Ensure all computations are complete before proceeding

print("Training completed. Final loss:", final_carry_state[4])
print("Training completed. Final loss:", updated_loss)
print("Training completed. Final cum_iter:", final_carry_state[0])


print("yoooooooooooooooooooooooooooooooooooooooo")
# Create a checkpoint manager or direct saver
ackp = ocp.StandardCheckpointer()
# Save parameters dictionary/tree to a path
ackp.save('/home/rohan/Desktop/FUN-Projects/MLFromScratch/jax_rnn_weights', final_carry_state,force=True)
ackp.wait_until_finished()
print("brooooooooooooooooooooooooooooooooooooooooo")

787000 total iterations
lasagnae JitTracer(int32[]) JitTracer(int32[]) JitTracer(int32[]) JitTracer(int32[])
loss this iteration JitTracer(bfloat16[])


TypeError: scan body function carry input and carry output must have equal types, but they differ:

  * the input carry component carry_state[3][0] has type bfloat16[64] but the corresponding output carry component has type bfloat16[32,64], so the shapes do not match;

  * the input carry component carry_state[3][1] has type bfloat16[32] but the corresponding output carry component has type bfloat16[32,32], so the shapes do not match;

  * the input carry component carry_state[3][2] has type bfloat16[1] but the corresponding output carry component has type bfloat16[32,1], so the shapes do not match;

  * the input carry component carry_state[7][0] has type bfloat16[64] but the corresponding output carry component has type bfloat16[32,64], so the shapes do not match;

  * the input carry component carry_state[7][1] has type bfloat16[32] but the corresponding output carry component has type bfloat16[32,32], so the shapes do not match;

  * the input carry component carry_state[7][2] has 

In [ ]:
print(len(updated_loss), "updated_loss length")
for loss in updated_loss:
    print("loss", loss)

78700 updated_loss length
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4448
loss 4448
loss 4448
loss 4448
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4480
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss 4448
loss

In [ ]:
print(min(updated_loss), "min loss")

4416 min loss
